# Lab 3 · Four portfolios, one universe

**Week 3 · Risk models and portfolio construction** · ≈ 90 minutes

Using a rolling 60-month window on the 25 Asia-Pacific portfolios, you will compare five constructions
out of sample: 1/N, sample minimum-variance, Ledoit–Wolf minimum-variance, equal risk contribution and
market-cap weights, and then reproduce the Black–Litterman starting point.

**Data.** Monthly returns from the Kenneth R. French Data Library (Tuck School of Business, Dartmouth), *Asia Pacific ex Japan* region: Australia, **Hong Kong**, New Zealand and Singapore, July 1990 onwards. The setup cell downloads it once and caches it in `finex_data/`.

**How this lab works.** Run the setup cell, then work top to bottom. Each exercise has a `# TODO` cell followed by an automatic check (✓ / ✗ with a hint). When every check passes, the last cell prints a **completion code**: paste it into the course page to complete the lab.

In [1]:
# FinEx Learning · lab setup. Run this cell first; it only defines helpers.
# Data: Kenneth R. French Data Library (Tuck School of Business, Dartmouth),
# Asia Pacific ex Japan series (Australia, Hong Kong, New Zealand, Singapore),
# downloaded on first use and cached next to the notebook.
import hashlib
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

FRENCH = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
CACHE = Path("finex_data")
CACHE.mkdir(exist_ok=True)


def _lines(name):
    path = CACHE / f"{name}.csv"
    if not path.exists():
        with urllib.request.urlopen(f"{FRENCH}{name}_CSV.zip", timeout=60) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
            path.write_bytes(archive.read(archive.namelist()[0]))
    return path.read_text(encoding="latin-1").splitlines()


def load_french(name, table=None, percent=True):
    """One monthly table from a French data library file, indexed by month.

    `table` is the title above the table (e.g. "Average Value Weighted Returns -- Monthly");
    None means the first monthly table. Returns are converted to decimals.
    """
    tables, title, current = {}, "", None
    for line in _lines(name):
        cells = [c.strip() for c in line.split(",")]
        if line.lstrip().startswith(","):
            current = []
            tables.setdefault(title or "first", (cells[1:], current))
            continue
        if current is not None and len(cells[0]) == 6 and cells[0].isdigit():
            current.append(cells)
            continue
        if line.strip():
            title, current = line.strip(), None
    key = table or next(k for k, (_, rows) in tables.items() if rows)
    header, rows = tables[key]
    frame = pd.DataFrame(
        [r[1:] for r in rows],
        columns=header,
        index=pd.PeriodIndex([r[0] for r in rows], freq="M"),
    ).astype(float).replace(-99.99, np.nan)
    return frame / 100 if percent else frame


def check(name, condition, hint):
    """Automatic check: passes quietly, or explains what to look at."""
    if not bool(condition):
        raise AssertionError(f"✗ {name}. Hint: {hint}")
    print(f"✓ {name}")


def completion_code(lab):
    code = hashlib.sha256(f"finex-quant-{lab}".encode()).hexdigest()[:6].upper()
    print(f"\nAll checks passed. Your completion code for the course page: FINEX-{lab.upper()}-{code}")


def annualise(returns):
    """Mean, volatility, Sharpe and t-stat of monthly excess returns, annualised."""
    r = returns.dropna()
    mean, vol = r.mean() * 12, r.std() * np.sqrt(12)
    return pd.Series({"mean": mean, "vol": vol, "sharpe": mean / vol,
                      "t_stat": r.mean() / r.std() * np.sqrt(len(r)), "months": len(r)})


print("Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.")


Setup complete. Data source: Kenneth R. French Data Library · Asia Pacific ex Japan.


In [2]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.covariance import LedoitWolf

returns = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Average Value Weighted Returns -- Monthly")
firms = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Number of Firms in Portfolios", percent=False)
avg_size = load_french("Asia_Pacific_ex_Japan_25_Portfolios_ME_BE-ME", "Average Firm Size", percent=False)
market_cap = firms * avg_size
cap_weights = market_cap.div(market_cap.sum(axis=1), axis=0)
WINDOW = 60

## Exercise 1 · Minimum variance and equal risk contribution

* Minimum variance: $w \propto \Sigma^{-1}\mathbf{1}$, scaled to sum to one (shorting allowed).
* Equal risk contribution: iterate $w_i \leftarrow w_i \sqrt{\bar{RC} / RC_i}$ with $RC_i = w_i(\Sigma w)_i$, renormalising each step.

In [3]:
def min_variance(cov):
    w = np.linalg.solve(cov, np.ones(len(cov)))
    return w / w.sum()

def equal_risk(cov, steps=500):
    w = np.ones(len(cov)) / len(cov)
    for _ in range(steps):
        rc = w * (cov @ w)
        w = w * np.sqrt(rc.mean() / rc)
        w = w / w.sum()
    return w

In [4]:
test_cov = returns.iloc[-WINDOW:].cov().values
rc = equal_risk(test_cov) * (test_cov @ equal_risk(test_cov))
check("Minimum-variance weights sum to one", np.isclose(min_variance(test_cov).sum(), 1), "divide by the sum")
check("Equal risk contributions are equal", np.allclose(rc, rc.mean(), rtol=1e-4), "keep iterating the multiplicative update")

✓ Minimum-variance weights sum to one
✓ Equal risk contributions are equal


## Exercise 2 · The out-of-sample horse race

Each month *t*, estimate on the previous 60 months, form the five portfolios, and record next month's
return and turnover (half the sum of absolute weight changes). Use `LedoitWolf().fit(window).covariance_`
for the shrunk matrix and `cap_weights` from the previous month.

In [5]:
records, turns, previous = [], [], {}
for i in range(WINDOW, len(returns)):
    window = returns.iloc[i - WINDOW:i]
    sample, shrunk = window.cov().values, LedoitWolf().fit(window.values).covariance_
    weights = {
        "1/N": np.ones(25) / 25,
        "sample min-var": min_variance(sample),
        "Ledoit-Wolf min-var": min_variance(shrunk),
        "equal risk": equal_risk(shrunk),
        "market cap": cap_weights.iloc[i - 1].values,
    }
    records.append({k: w @ returns.iloc[i].values for k, w in weights.items()})
    if previous:
        turns.append({k: np.abs(w - previous[k]).sum() / 2 for k, w in weights.items()})
    previous = weights
oos = pd.DataFrame(records, index=returns.index[WINDOW:])
turnover = pd.DataFrame(turns).mean()
summary = pd.DataFrame({"oos_vol": oos.std() * np.sqrt(12), "monthly_turnover": turnover,
                        "gross_exposure_now": {k: np.abs(w).sum() for k, w in previous.items()}})
summary.round(3)

,oos_vol,monthly_turnover,gross_exposure_now
1/N,0.211,0.000,1.000
sample min-var,0.217,0.846,7.340
Ledoit-Wolf min-var,0.182,0.258,4.974
equal risk,0.208,0.004,1.000
market cap,0.197,0.018,1.000


In [6]:
vol = oos.std() * np.sqrt(12)
check("Five constructions evaluated", isinstance(oos, pd.DataFrame) and oos.shape[1] == 5, "one column per portfolio")
check("Shrinkage beats the sample estimate out of sample", vol["Ledoit-Wolf min-var"] < vol["sample min-var"],
      "use LedoitWolf().fit(window.values).covariance_ for the shrunk version")
check("Shrinkage also cuts turnover", turnover["Ledoit-Wolf min-var"] < turnover["sample min-var"],
      "turnover = half the sum of absolute weight changes")

✓ Five constructions evaluated
✓ Shrinkage beats the sample estimate out of sample
✓ Shrinkage also cuts turnover


## Exercise 3 · The Black–Litterman starting point

With risk aversion λ = 3, the implied excess returns are $\Pi = \lambda \Sigma w_{mkt}$. Show that an
unconstrained mean–variance investor who believes Π holds exactly the market portfolio. Then solve a
long-only version with a 10% cap per asset using `cvxpy`.

In [7]:
import cvxpy as cp

LAMBDA = 3.0
sigma = LedoitWolf().fit(returns.iloc[-WINDOW:].values).covariance_ * 12
w_mkt = cap_weights.iloc[-1].values
implied = LAMBDA * sigma @ w_mkt
w_unconstrained = np.linalg.solve(LAMBDA * sigma, implied)
w = cp.Variable(25)
problem = cp.Problem(cp.Maximize(implied @ w - LAMBDA / 2 * cp.quad_form(w, cp.psd_wrap(sigma))),
                     [cp.sum(w) == 1, w >= 0, w <= 0.10])
problem.solve()
w_long_only = w.value
pd.Series(w_long_only, index=returns.columns).nlargest(8).round(3)

ME4 BM5     0.100
ME5 BM3     0.100
ME5 BM2     0.100
ME5 BM4     0.100
ME4 BM3     0.100
BIG HiBM    0.100
BIG LoBM    0.100
ME4 BM2     0.076
dtype: float64

In [8]:
check("Implied returns reproduce the market portfolio", np.allclose(w_unconstrained, w_mkt, atol=1e-8),
      "Π = λΣw_mkt, then solve (λΣ) w = Π")
check("Long-only solution respects the constraints",
      w_long_only is not None and np.isclose(w_long_only.sum(), 1, atol=1e-4) and w_long_only.min() > -1e-6 and w_long_only.max() < 0.10 + 1e-5,
      "constraints: sum to one, non-negative, at most 10%")
completion_code("lab3")

✓ Implied returns reproduce the market portfolio
✓ Long-only solution respects the constraints

All checks passed. Your completion code for the course page: FINEX-LAB3-E0D555


## Reflect

* The sample minimum-variance portfolio is the 'optimal' one in-sample. Why is it the most volatile out of sample?
* Look at `gross_exposure_now`. Would a risk committee approve the sample portfolio?
* Which construction would you take into the capstone, and why?